In [1]:
import numpy as np
from common import client, ask

EMBED_MODEL = "gemini-embedding-001"   

docs = [
    "성능점검기록부에는 사고 이력, 주요 부품 상태, 주행거리가 나온다. 구매 전 꼭 받아서 확인한다.",
    "침수차는 안전벨트를 끝까지 당겨 진흙이나 물자국이 있는지, 실내에서 곰팡이 냄새가 나는지 본다.",
    "명의 이전에는 양도인의 인감증명서와 양수인의 신분증, 자동차 보험 가입이 필요하다.",
    "연간 주행거리는 보통 1만 5천~2만 km이다. 연식에 비해 너무 적거나 많으면 이유를 물어본다.",
    "시세보다 크게 싼 매물은 허위 매물일 수 있으므로 실차를 확인하기 전에 계약금을 보내지 않는다.",
]

def embed(texts):
    r = client.models.embed_content(model=EMBED_MODEL, contents=texts)
    return np.array([e.values for e in r.embeddings])

doc_vecs = embed(docs)                        # 1. 준비: 한 번만

def search(question, k=3):
    q = embed([question])[0]
    # 코사인 유사도를 계산. 저장된 문서 중 질문과 가장 가까운 문서를 찾기 위해 
    sims = doc_vecs @ q / (np.linalg.norm(doc_vecs, axis=1) * np.linalg.norm(q))
    # 유사도가 높은 순으로 정렬 후 k개를 선택
    top = np.argsort(sims)[::-1][:k]          # 2. 검색: 유사도 높은 순
    return [(docs[i], round(float(sims[i]), 3)) for i in top]

question = "물에 잠긴 차인지 어떻게 알아요?"
found = search(question)
for text, sim in found:
    print(sim, text)

context = "\n".join(f"[{i+1}] {t}" for i, (t, _) in enumerate(found))
r = ask(f"""아래 문서에 있는 내용으로만 답하라. 문서에 없으면 '자료에 없습니다'라고 답한다.
답 끝에 참고한 문서 번호를 적는다.

{context}

질문: {question}""", temperature=0)        # 3. 답변
print(r.text)

0.795 침수차는 안전벨트를 끝까지 당겨 진흙이나 물자국이 있는지, 실내에서 곰팡이 냄새가 나는지 본다.
0.64 시세보다 크게 싼 매물은 허위 매물일 수 있으므로 실차를 확인하기 전에 계약금을 보내지 않는다.
0.623 성능점검기록부에는 사고 이력, 주요 부품 상태, 주행거리가 나온다. 구매 전 꼭 받아서 확인한다.
  (gemini-3.1-flash-lite 응답)
안전벨트를 끝까지 당겨 진흙이나 물자국이 있는지 확인하고, 실내에서 곰팡이 냄새가 나는지 확인합니다. [1]
